## PatchTST

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/DaTking4/ml-final-project-walmart-recruiting.git"], check=True)
    %cd ml-final-project-walmart-recruiting
    %pip install -q -r requirements.txt

    import os
    from google.colab import userdata
    os.environ["DAGSHUB_USER_TOKEN"] = userdata.get("DAGSHUB_TOKEN")
    os.environ["WANDB_API_KEY"]      = userdata.get("WANDB_API_KEY")
    os.environ["KAGGLE_API_TOKEN"]   = userdata.get("KAGGLE_API_TOKEN")

    %pip install -q kaggle
    import os; os.makedirs("data", exist_ok=True)
    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data/ --quiet
    !unzip -q -o data/walmart-recruiting-store-sales-forecasting.zip -d data/

print("Running in:", "Google Colab" if IN_COLAB else "Local environment")


### 1. Setup and Imports

In [ ]:
import os
import sys
import importlib
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))
os.environ.setdefault("NIXTLA_ID_AS_COL", "1")

repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / "src").exists():
        sys.path.insert(0, str(repo_root)); break
    repo_root = repo_root.parent
else:
    raise FileNotFoundError("Could not locate repo root containing 'src'.")

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature
import wandb

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

import src.mlflow_setup as mlflow_setup
importlib.reload(mlflow_setup)
init_tracking = mlflow_setup.init_tracking

from src.data_loading import load_merged
from src.transforms import apply_shared_features
from src.validation import time_based_split
from src.metrics import wmae_from_df
from src.pipeline.dlinear_pipeline import to_long_format
from src.pipeline.patchtst_pipeline import PatchTSTForecastPipeline
from src.training_diagnostics import GradientNormLogger, log_gradient_diagnostics, strip_neuralforecast_callbacks

init_tracking()
assert "dagshub.com" in mlflow.get_tracking_uri(), mlflow.get_tracking_uri()
if mlflow.active_run(): mlflow.end_run()

BLUE, PINK, PURPLE, RED, GREEN = "#7196C7", "#AE737D", "#705588", "#7E3838", "#5E9D74"
REGIME_COLORS = {"underfit": PURPLE, "balanced": BLUE, "overfit": RED}
STATUS_COLORS = {"good": GREEN, "underfit": PURPLE, "overfit": RED}

print("Setup complete.")


### 2. Configuration

In [ ]:
init_tracking()
assert "dagshub.com" in mlflow.get_tracking_uri(), mlflow.get_tracking_uri()

EXPERIMENT_NAME = "PatchTST_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

WANDB_ENTITY  = "dkhak22-free-university-of-tbilisi-"
WANDB_PROJECT = "walmart-sales-forecasting"

# Detect GPU so the same notebook works locally (CPU) and on Colab (T4/A100).
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES     = 1
print(f"Accelerator: {ACCELERATOR}")

CONFIG = {
    "input_size":     52,
    "horizon":        26,
    "patch_len":      16,
    "stride":         8,
    "hidden_size":    128,
    "linear_hidden_size": 256,
    "n_heads":        8,
    "encoder_layers": 3,
    "dropout":        0.2,
    "fc_dropout":     0.2,
    "batch_size":     64,
    "learning_rate":  1e-4,
    "max_steps":      500,
    "random_seed":    42,
}

FREQ      = "W-FRI"
MODEL_COL = "PatchTST"

CONFIG


### 3. Load Data

In [ ]:
train_df, test_df = load_merged()
print(f"train_df: {train_df.shape}")
print(f"test_df:  {test_df.shape}")
CONFIG["horizon"] = test_df["Date"].nunique()
train_df.head()


### 4. Shared Preprocessing and Feature Engineering

In [ ]:
train_feat = apply_shared_features(train_df)
print(f"Columns after shared feature engineering: {train_feat.shape[1]}")
train_feat.head()


### 5. Model-Specific Format

In [ ]:
long_df = to_long_format(train_feat, include_target=True)

print(f"long_df shape: {long_df.shape}")
print(f"Number of time series: {long_df['unique_id'].nunique()}")
print("\nColumns used by PatchTST:")
print(long_df.columns.tolist())


### 6. Feature Selection

In [ ]:
exogenous_support = {
    "future_exogenous":    getattr(PatchTST, "EXOGENOUS_FUTR", False),
    "historical_exogenous": getattr(PatchTST, "EXOGENOUS_HIST", False),
    "static_exogenous":    getattr(PatchTST, "EXOGENOUS_STAT", False),
}
display(pd.Series(exogenous_support, name="PatchTST exogenous support"))

PATCHTST_FEATURE_DECISION = {
    "feature_set": "target_history_only",
    "uses_exogenous_features": False,
    "used_model_columns": "unique_id, ds, y",
    "reason": (
        "PatchTST uses channel-independent patched self-attention: each "
        "Store-Dept series is divided into non-overlapping (or strided) "
        "patches and processed by a shared Transformer encoder. The model "
        "captures local temporal patterns within patches and global patterns "
        "across patches, without needing exogenous features."
    ),
}
PATCHTST_FEATURE_DECISION


### 7. Time-Series and Window Setup

In [ ]:
train_part, valid_part = time_based_split(train_feat, valid_weeks=CONFIG["horizon"])

print(f"Train part: {train_part['Date'].min().date()} -> {train_part['Date'].max().date()}")
print(f"Valid part: {valid_part['Date'].min().date()} -> {valid_part['Date'].max().date()}")
print(f"\nInput size:   {CONFIG['input_size']} weeks")
print(f"Horizon:      {CONFIG['horizon']} weeks")
print(f"Patch length: {CONFIG['patch_len']}  Stride: {CONFIG['stride']}")
n_patches = (CONFIG["input_size"] - CONFIG["patch_len"]) // CONFIG["stride"] + 1
print(f"Num patches:  {n_patches}")

series_lengths = long_df.groupby("unique_id").size()
PATCHTST_SERIES_LENGTH = int(series_lengths.max())
patchtst_ids = series_lengths[series_lengths == PATCHTST_SERIES_LENGTH].index
patchtst_df  = long_df[long_df["unique_id"].isin(patchtst_ids)].copy()

print(f"\nUsing {patchtst_df['unique_id'].nunique()} complete-history series")
print(f"Dropping {long_df['unique_id'].nunique() - patchtst_df['unique_id'].nunique()} short/ragged series")

holiday_lookup = train_feat.assign(
    unique_id=train_feat["Store"].astype(str) + "_" + train_feat["Dept"].astype(str),
    ds=train_feat["Date"],
)[["unique_id", "ds", "IsHoliday"]].drop_duplicates()
holiday_lookup["IsHoliday"] = holiday_lookup["IsHoliday"].fillna(False).astype(bool)

def ensure_unique_id_column(df):
    if "unique_id" not in df.columns and df.index.name == "unique_id":
        return df.reset_index()
    return df

def fit_gap_pct(train_wmae, val_wmae):
    if pd.isna(train_wmae) or train_wmae == 0: return float("nan")
    return ((val_wmae - train_wmae) / train_wmae) * 100

def classify_fit_status(train_wmae, val_wmae):
    gap = fit_gap_pct(train_wmae, val_wmae)
    if pd.isna(gap): return "unknown"
    return "overfit" if gap > 25 else "underfit" if gap < -10 else "good"

def save_nf_model(nf, path, overwrite=True):
    if not hasattr(nf, "prediction_intervals"): nf.prediction_intervals = None
    if not hasattr(nf, "_cs_df"): nf._cs_df = None
    nf.save(path=path, overwrite=overwrite, save_dataset=True)

print("Window setup complete.")


### 8. Sanity Check

In [ ]:
tiny_ids = patchtst_df["unique_id"].unique()[:5]
tiny_df  = patchtst_df[patchtst_df["unique_id"].isin(tiny_ids)]

sanity_model = PatchTST(
    h=CONFIG["horizon"],
    input_size=CONFIG["input_size"],
    patch_len=16, stride=8,
    hidden_size=32, linear_hidden_size=64,
    n_heads=4, encoder_layers=1,
    dropout=0.0, fc_dropout=0.0,
    max_steps=2,
    batch_size=8,
    learning_rate=CONFIG["learning_rate"],
    random_seed=CONFIG["random_seed"],
    start_padding_enabled=True,
    scaler_type="robust",
    accelerator="cpu",
    devices=1,
)

sanity_nf = NeuralForecast(models=[sanity_model], freq=FREQ)
sanity_cv = sanity_nf.cross_validation(df=tiny_df, n_windows=1, step_size=CONFIG["horizon"])
sanity_cv = ensure_unique_id_column(sanity_cv)

assert len(sanity_cv) == len(tiny_ids) * CONFIG["horizon"], \
    f"Expected {len(tiny_ids) * CONFIG['horizon']} rows, got {len(sanity_cv)}"
assert MODEL_COL in sanity_cv.columns, f"Column '{MODEL_COL}' missing from CV output"

print("Sanity check passed.")
display(sanity_cv.head())


### 9. Baseline Run

In [ ]:
with mlflow.start_run(run_name="PatchTST_Baseline") as run:
    wandb.init(
        entity=WANDB_ENTITY, project=WANDB_PROJECT,
        name="PatchTST_Baseline", group="PatchTST", job_type="baseline",
        tags=["PatchTST", "baseline", "target_history_only"],
        config={**CONFIG, **PATCHTST_FEATURE_DECISION,
                "patchtst_series_length": PATCHTST_SERIES_LENGTH,
                "patchtst_n_series": patchtst_df["unique_id"].nunique(),
                "accelerator": ACCELERATOR},
        reinit=True,
    )
    try:
        gradient_callback = GradientNormLogger(log_every_n_steps=10)

        baseline_model = PatchTST(
            h=CONFIG["horizon"],
            input_size=CONFIG["input_size"],
            patch_len=CONFIG["patch_len"],
            stride=CONFIG["stride"],
            hidden_size=CONFIG["hidden_size"],
            linear_hidden_size=CONFIG["linear_hidden_size"],
            n_heads=CONFIG["n_heads"],
            encoder_layers=CONFIG["encoder_layers"],
            dropout=CONFIG["dropout"],
            fc_dropout=CONFIG["fc_dropout"],
            max_steps=CONFIG["max_steps"],
            batch_size=CONFIG["batch_size"],
            learning_rate=CONFIG["learning_rate"],
            random_seed=CONFIG["random_seed"],
            start_padding_enabled=True,
            scaler_type="robust",
            accelerator=ACCELERATOR,
            devices=DEVICES,
            callbacks=[gradient_callback],
        )

        baseline_nf = NeuralForecast(models=[baseline_model], freq=FREQ)
        cv_df = baseline_nf.cross_validation(df=patchtst_df, n_windows=1, step_size=CONFIG["horizon"])
        cv_df = ensure_unique_id_column(cv_df)
        cv_df = cv_df.merge(holiday_lookup, on=["unique_id", "ds"], how="left")
        cv_df["IsHoliday"] = cv_df["IsHoliday"].fillna(False).astype(bool)

        val_wmae = wmae_from_df(cv_df, y_true_col="y", y_pred_col=MODEL_COL, holiday_col="IsHoliday")

        train_pred_df = baseline_nf.predict_insample(step_size=CONFIG["horizon"])
        train_pred_df = ensure_unique_id_column(train_pred_df)
        train_pred_df = train_pred_df.merge(holiday_lookup, on=["unique_id", "ds"], how="left")
        train_pred_df["IsHoliday"] = train_pred_df["IsHoliday"].fillna(False).astype(bool)
        train_wmae = wmae_from_df(train_pred_df, y_true_col="y", y_pred_col=MODEL_COL, holiday_col="IsHoliday")

        gap    = fit_gap_pct(train_wmae, val_wmae)
        status = classify_fit_status(train_wmae, val_wmae)

        gradient_metrics = log_gradient_diagnostics(
            gradient_callback, model_name="PatchTST", run_label="baseline",
            mlflow_module=mlflow, wandb_module=wandb,
        )

        mlflow.log_params({**CONFIG, "regime": "baseline", "accelerator": ACCELERATOR,
                           "feature_set": PATCHTST_FEATURE_DECISION["feature_set"]})
        mlflow.log_metrics({"train_wmae": train_wmae, "val_wmae": val_wmae, "gap_pct": gap})
        mlflow.log_param("fit_status", status)

        wandb.log({"train_wmae": train_wmae, "val_wmae": val_wmae, "gap_pct": gap, "fit_status": status})

        print(f"Baseline  train WMAE: {train_wmae:,.2f}")
        print(f"Baseline  val   WMAE: {val_wmae:,.2f}")
        print(f"Gap: {gap:.1f}%  Status: {status}")
    finally:
        wandb.finish()


### 10. Hyperparameters

In [ ]:
param_grid = [
    {"input_size": 52, "patch_len": 24, "stride": 12, "hidden_size": 32,  "linear_hidden_size": 64,   "n_heads": 4,  "encoder_layers": 1, "dropout": 0.3, "fc_dropout": 0.3, "max_steps": 50,   "learning_rate": 1e-4, "batch_size": 128, "label": "underfit_1", "regime": "underfit"},
    {"input_size": 52, "patch_len": 24, "stride": 12, "hidden_size": 32,  "linear_hidden_size": 64,   "n_heads": 4,  "encoder_layers": 1, "dropout": 0.4, "fc_dropout": 0.4, "max_steps": 100,  "learning_rate": 1e-4, "batch_size": 128, "label": "underfit_2", "regime": "underfit"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 32,  "linear_hidden_size": 64,   "n_heads": 4,  "encoder_layers": 1, "dropout": 0.4, "fc_dropout": 0.4, "max_steps": 50,   "learning_rate": 1e-4, "batch_size": 128, "label": "underfit_3", "regime": "underfit"},
    {"input_size": 52, "patch_len": 24, "stride": 12, "hidden_size": 64,  "linear_hidden_size": 128,  "n_heads": 4,  "encoder_layers": 1, "dropout": 0.3, "fc_dropout": 0.3, "max_steps": 50,   "learning_rate": 1e-4, "batch_size": 128, "label": "underfit_4", "regime": "underfit"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 32,  "linear_hidden_size": 64,   "n_heads": 4,  "encoder_layers": 2, "dropout": 0.4, "fc_dropout": 0.4, "max_steps": 100,  "learning_rate": 1e-4, "batch_size": 128, "label": "underfit_5", "regime": "underfit"},
    {"input_size": 52, "patch_len": 24, "stride": 12, "hidden_size": 64,  "linear_hidden_size": 128,  "n_heads": 4,  "encoder_layers": 1, "dropout": 0.4, "fc_dropout": 0.4, "max_steps": 100,  "learning_rate": 1e-4, "batch_size": 128, "label": "underfit_6", "regime": "underfit"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 300,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_1",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_2",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 16, "encoder_layers": 3, "dropout": 0.1, "fc_dropout": 0.1, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 32,  "label": "balanced_3",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 24, "stride": 12, "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 300,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_4",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 300,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_5",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_6",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 512,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.1, "fc_dropout": 0.1, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_7",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 4, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_8",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 256, "linear_hidden_size": 512,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 300,  "learning_rate": 1e-4, "batch_size": 32,  "label": "balanced_9",  "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 256, "linear_hidden_size": 512,  "n_heads": 16, "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 32,  "label": "balanced_10", "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 64,  "linear_hidden_size": 128,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.1, "fc_dropout": 0.1, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 128, "label": "balanced_11", "regime": "balanced"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 256, "linear_hidden_size": 512,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.1, "fc_dropout": 0.1, "max_steps": 300,  "learning_rate": 1e-4, "batch_size": 32,  "label": "balanced_12", "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 5e-4, "batch_size": 64,  "label": "balanced_13", "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 128, "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 3, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_14", "regime": "balanced"},
    {"input_size": 52, "patch_len": 24, "stride": 8,  "hidden_size": 256, "linear_hidden_size": 512,  "n_heads": 16, "encoder_layers": 3, "dropout": 0.1, "fc_dropout": 0.1, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 32,  "label": "balanced_15", "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 64,  "linear_hidden_size": 256,  "n_heads": 8,  "encoder_layers": 4, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 128, "label": "balanced_16", "regime": "balanced"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 256, "linear_hidden_size": 512,  "n_heads": 8,  "encoder_layers": 4, "dropout": 0.2, "fc_dropout": 0.2, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 32,  "label": "balanced_17", "regime": "balanced"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 128, "linear_hidden_size": 512,  "n_heads": 8,  "encoder_layers": 4, "dropout": 0.1, "fc_dropout": 0.1, "max_steps": 500,  "learning_rate": 1e-4, "batch_size": 64,  "label": "balanced_18", "regime": "balanced"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 512, "linear_hidden_size": 1024, "n_heads": 16, "encoder_layers": 6, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 1000, "learning_rate": 1e-4, "batch_size": 32,  "label": "overfit_1",  "regime": "overfit"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 512, "linear_hidden_size": 1024, "n_heads": 16, "encoder_layers": 6, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 2000, "learning_rate": 1e-4, "batch_size": 32,  "label": "overfit_2",  "regime": "overfit"},
    {"input_size": 52, "patch_len": 16, "stride": 8,  "hidden_size": 512, "linear_hidden_size": 1024, "n_heads": 16, "encoder_layers": 6, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 2000, "learning_rate": 1e-4, "batch_size": 32,  "label": "overfit_3",  "regime": "overfit"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 256, "linear_hidden_size": 1024, "n_heads": 16, "encoder_layers": 6, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 1000, "learning_rate": 1e-4, "batch_size": 32,  "label": "overfit_4",  "regime": "overfit"},
    {"input_size": 52, "patch_len": 8,  "stride": 4,  "hidden_size": 512, "linear_hidden_size": 1024, "n_heads": 16, "encoder_layers": 4, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 2000, "learning_rate": 5e-5, "batch_size": 32,  "label": "overfit_5",  "regime": "overfit"},
    {"input_size": 52, "patch_len": 16, "stride": 4,  "hidden_size": 512, "linear_hidden_size": 1024, "n_heads": 32, "encoder_layers": 6, "dropout": 0.0, "fc_dropout": 0.0, "max_steps": 1000, "learning_rate": 1e-4, "batch_size": 32,  "label": "overfit_6",  "regime": "overfit"},
]

print(f"Total configs: {len(param_grid)}")
regime_counts = {r: sum(1 for c in param_grid if c['regime'] == r) for r in ['underfit','balanced','overfit']}
print("By regime:", regime_counts)


### 11. PatchTST Experiments

In [ ]:
results     = []
cv_by_label = {}

best_val_wmae = float("inf")
best_run_id   = None
best_label    = None
best_nf_path  = None

with mlflow.start_run(run_name="PatchTST_HyperparamSweep") as parent_run:
    mlflow.log_param("n_configs",           len(param_grid))
    mlflow.log_param("model",               "PatchTST")
    mlflow.log_param("feature_set",         PATCHTST_FEATURE_DECISION["feature_set"])
    mlflow.log_param("patchtst_series_length", PATCHTST_SERIES_LENGTH)
    mlflow.log_param("patchtst_n_series",   patchtst_df["unique_id"].nunique())
    mlflow.log_param("accelerator",         ACCELERATOR)

    for original_params in param_grid:
        params = original_params.copy()
        label  = params.pop("label")
        regime = params.pop("regime")

        with mlflow.start_run(run_name=f"PatchTST_{label}", nested=True) as nested_run:
            wandb.init(
                entity=WANDB_ENTITY, project=WANDB_PROJECT,
                name=f"PatchTST_{label}", group="PatchTST",
                job_type="hyperparameter_sweep",
                tags=["PatchTST", regime, "target_history_only"],
                config={**params, "regime": regime, "accelerator": ACCELERATOR,
                        "feature_set": PATCHTST_FEATURE_DECISION["feature_set"]},
                reinit=True,
            )
            try:
                gradient_callback = GradientNormLogger(log_every_n_steps=10)

                model = PatchTST(
                    h=CONFIG["horizon"],
                    input_size=params["input_size"],
                    patch_len=params["patch_len"],
                    stride=params["stride"],
                    hidden_size=params["hidden_size"],
                    linear_hidden_size=params["linear_hidden_size"],
                    n_heads=params["n_heads"],
                    encoder_layers=params["encoder_layers"],
                    dropout=params["dropout"],
                    fc_dropout=params["fc_dropout"],
                    max_steps=params["max_steps"],
                    batch_size=params["batch_size"],
                    learning_rate=params["learning_rate"],
                    random_seed=CONFIG["random_seed"],
                    start_padding_enabled=True,
                    scaler_type="robust",
                    accelerator=ACCELERATOR,
                    devices=DEVICES,
                    callbacks=[gradient_callback],
                )

                nf = NeuralForecast(models=[model], freq=FREQ)
                cv_df = nf.cross_validation(df=patchtst_df, n_windows=1, step_size=CONFIG["horizon"])
                cv_df = ensure_unique_id_column(cv_df)
                cv_df = cv_df.merge(holiday_lookup, on=["unique_id", "ds"], how="left")
                cv_df["IsHoliday"] = cv_df["IsHoliday"].fillna(False).astype(bool)

                val_wmae = wmae_from_df(cv_df, y_true_col="y", y_pred_col=MODEL_COL, holiday_col="IsHoliday")

                train_pred_df = nf.predict_insample(step_size=CONFIG["horizon"])
                train_pred_df = ensure_unique_id_column(train_pred_df)
                train_pred_df = train_pred_df.merge(holiday_lookup, on=["unique_id", "ds"], how="left")
                train_pred_df["IsHoliday"] = train_pred_df["IsHoliday"].fillna(False).astype(bool)
                train_wmae = wmae_from_df(train_pred_df, y_true_col="y", y_pred_col=MODEL_COL, holiday_col="IsHoliday")

                gap    = fit_gap_pct(train_wmae, val_wmae)
                status = classify_fit_status(train_wmae, val_wmae)

                cv_by_label[label] = cv_df.copy()

                gradient_metrics = log_gradient_diagnostics(
                    gradient_callback, model_name="PatchTST", run_label=label,
                    mlflow_module=mlflow, wandb_module=wandb,
                )

                mlflow.log_params({**params, "regime": regime, "label": label,
                                   "feature_set": PATCHTST_FEATURE_DECISION["feature_set"],
                                   "accelerator": ACCELERATOR})
                mlflow.log_metrics({"train_wmae": train_wmae, "val_wmae": val_wmae, "gap_pct": gap})
                mlflow.log_param("fit_status", status)

                wandb.log({"train_wmae": train_wmae, "val_wmae": val_wmae,
                           "gap_pct": gap, "fit_status": status})

                results.append({
                    "label": label, "regime": regime, **params,
                    "feature_set": PATCHTST_FEATURE_DECISION["feature_set"],
                    "train_wmae": train_wmae, "val_wmae": val_wmae,
                    "gap_pct": gap, "status": status,
                    **gradient_metrics,
                    "run_id": nested_run.info.run_id,
                })

                if val_wmae < best_val_wmae:
                    best_val_wmae = val_wmae
                    best_run_id   = nested_run.info.run_id
                    best_label    = label
                    best_nf_path  = f"artifacts/patchtst_{label}"
                    save_nf_model(nf, path=best_nf_path, overwrite=True)
                    mlflow.log_artifacts(best_nf_path, artifact_path="nf_model")
                    print(f"New best: {best_label} | train={train_wmae:,.2f} | val={best_val_wmae:,.2f} | gap={gap:.1f}% ({status})")

            finally:
                wandb.finish()


### 12. Results

In [ ]:
results_df = pd.DataFrame(results).sort_values("val_wmae").reset_index(drop=True)

display_cols = [
    "label", "regime", "status",
    "patch_len", "stride", "hidden_size", "n_heads", "encoder_layers",
    "dropout", "max_steps", "learning_rate", "batch_size",
    "grad_total_norm_mean", "grad_total_norm_max",
    "train_wmae", "val_wmae", "gap_pct",
]
available_cols = [c for c in display_cols if c in results_df.columns]
display(results_df[available_cols].head(15))

os.makedirs("reports", exist_ok=True)
results_path = "reports/patchtst_results.csv"
results_df.to_csv(results_path, index=False)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(results_path)
    mlflow.log_metric("best_val_wmae", best_val_wmae)

print(f"\nBest: {best_label}  val_wmae={best_val_wmae:,.2f}")


### 13. Plots

In [ ]:
os.makedirs("Plots", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for status, sdf in results_df.groupby("status"):
    axes[0].scatter(sdf["train_wmae"], sdf["val_wmae"],
                    c=STATUS_COLORS.get(status, BLUE), alpha=0.75, label=status)
lo = float(results_df[["train_wmae", "val_wmae"]].min().min())
hi = float(results_df[["train_wmae", "val_wmae"]].max().max())
axes[0].plot([lo, hi], [lo, hi], "k--", alpha=0.4)
axes[0].set_xlabel("Train WMAE"); axes[0].set_ylabel("Val WMAE")
axes[0].set_title("PatchTST — Bias-Variance Tradeoff")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for regime, rdf in results_df.groupby("regime"):
    axes[1].scatter(rdf["hidden_size"], rdf["val_wmae"],
                    c=REGIME_COLORS.get(regime, BLUE), alpha=0.75, label=regime)
axes[1].set_xlabel("hidden_size"); axes[1].set_ylabel("Val WMAE")
axes[1].set_title("PatchTST — Val WMAE vs hidden_size")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = "Plots/patchtst_sweep.png"
plt.savefig(plot_path, dpi=200); plt.show()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(plot_path)


### 14. Error Analysis

In [ ]:
best_cv_df = cv_by_label[best_label].copy()
best_cv_df["abs_error"] = (best_cv_df["y"] - best_cv_df[MODEL_COL]).abs()
best_cv_df[["Store", "Dept"]] = best_cv_df["unique_id"].str.split("_", n=1, expand=True)

worst_store_dept = (
    best_cv_df.groupby(["Store", "Dept"])["abs_error"]
    .mean().sort_values(ascending=False).head(15)
)
display(worst_store_dept)

holiday_error = best_cv_df.groupby("IsHoliday")["abs_error"].mean()
display(holiday_error)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metric("holiday_week_mae",    float(holiday_error.get(True,  float("nan"))))
    mlflow.log_metric("nonholiday_week_mae", float(holiday_error.get(False, float("nan"))))


### 15. Error Plots

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
worst_store_dept.sort_values().plot(kind="barh", ax=ax, color=RED)
ax.set_xlabel("Mean Absolute Error")
ax.set_title("PatchTST — Worst Store/Dept Validation Errors")
ax.grid(True, alpha=0.3)
plt.tight_layout()
error_plot_path = "Plots/patchtst_worst_store_dept.png"
plt.savefig(error_plot_path, dpi=200); plt.show()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(error_plot_path)


### 16. Best Model

In [ ]:
print("Best label:",           best_label)
print("Best run id:",          best_run_id)
print("Best validation WMAE:", best_val_wmae)

assert best_label  is not None
assert best_run_id is not None

best_row = results_df[results_df["label"] == best_label].iloc[0]
best_params = {
    "input_size":         int(best_row["input_size"]),
    "patch_len":          int(best_row["patch_len"]),
    "stride":             int(best_row["stride"]),
    "hidden_size":        int(best_row["hidden_size"]),
    "linear_hidden_size": int(best_row["linear_hidden_size"]),
    "n_heads":            int(best_row["n_heads"]),
    "encoder_layers":     int(best_row["encoder_layers"]),
    "dropout":            float(best_row["dropout"]),
    "fc_dropout":         float(best_row["fc_dropout"]),
    "max_steps":          int(best_row["max_steps"]),
    "learning_rate":      float(best_row["learning_rate"]),
    "batch_size":         int(best_row["batch_size"]),
}
print("Best params:"); display(best_params)

fallback_by_id = (
    long_df.sort_values("ds").groupby("unique_id")["y"]
    .last().astype(float).to_dict()
)
global_fallback = float(long_df["y"].median())

final_gradient_callback = GradientNormLogger(log_every_n_steps=10)

final_model = PatchTST(
    h=CONFIG["horizon"],
    input_size=best_params["input_size"],
    patch_len=best_params["patch_len"],
    stride=best_params["stride"],
    hidden_size=best_params["hidden_size"],
    linear_hidden_size=best_params["linear_hidden_size"],
    n_heads=best_params["n_heads"],
    encoder_layers=best_params["encoder_layers"],
    dropout=best_params["dropout"],
    fc_dropout=best_params["fc_dropout"],
    max_steps=best_params["max_steps"],
    batch_size=best_params["batch_size"],
    learning_rate=best_params["learning_rate"],
    random_seed=CONFIG["random_seed"],
    start_padding_enabled=True,
    scaler_type="robust",
    accelerator=ACCELERATOR,
    devices=DEVICES,
    callbacks=[final_gradient_callback],
)

final_nf = NeuralForecast(models=[final_model], freq=FREQ)
final_nf.fit(df=patchtst_df)

with mlflow.start_run(run_id=best_run_id):
    log_gradient_diagnostics(
        final_gradient_callback, model_name="PatchTST",
        run_label=f"final_{best_label}",
        mlflow_module=mlflow, wandb_module=None,
    )
    mlflow.log_param("registered_model_name", "PatchTST_WalmartForecast")
    mlflow.log_metric("best_val_wmae", best_val_wmae)

for m in final_nf.models:
    strip_neuralforecast_callbacks(m)

os.makedirs("artifacts", exist_ok=True)
best_nf_path = f"artifacts/patchtst_final_{best_label}"
save_nf_model(final_nf, path=best_nf_path, overwrite=True)
print("Final NeuralForecast model saved to:", best_nf_path)

pipeline_model = PatchTSTForecastPipeline(
    model_col=MODEL_COL,
    fallback_by_id=fallback_by_id,
    global_fallback=global_fallback,
)

class _Ctx:
    artifacts = {"nf_model_dir": best_nf_path}

_temp = PatchTSTForecastPipeline(model_col=MODEL_COL,
                                  fallback_by_id=fallback_by_id,
                                  global_fallback=global_fallback)
_temp.load_context(_Ctx())
sample_output = _temp.predict(_Ctx(), test_df)
signature = infer_signature(test_df, sample_output)
display(sample_output.head())

with mlflow.start_run(run_id=best_run_id):
    logged_model_info = mlflow.pyfunc.log_model(
        artifact_path="pipeline",
        python_model=pipeline_model,
        artifacts={"nf_model_dir": best_nf_path},
        code_paths=[str(repo_root / "src")],
        signature=signature,
        input_example=test_df.head(20),
        registered_model_name="PatchTST_WalmartForecast",
    )

model_uri = logged_model_info.model_uri
print("Logged model URI:", model_uri)


### 17. Test Loading

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_preds = loaded_model.predict(test_df)

print(type(loaded_preds))
print(loaded_preds.shape)
display(loaded_preds.head())
